In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

import doralite
import gfdl_utils.core as gu
import CM4Xutils
import numpy as np
import xarray as xr
import xgcm

import gsw, xwmt, xhistogram
import zarr
from dask.diagnostics import ProgressBar

In [3]:
import cmocean
import matplotlib.colors as colors
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.legend_handler import HandlerLine2D, HandlerTuple
plt.rcParams.update({'font.size': 12})

In [4]:
from common import *
grids = load_datasets()

sim = 'CM4Xp125_forced'
grid = grids[sim]
ds = grid._ds

Inferring Z grid coordinate: depth `z_`
Inferring Z grid coordinate: depth `z_`


In [5]:
tracer = "cfc11"
ds["volcello"] = ds.thkcello.fillna(0.)*ds.areacello.fillna(0.)
tracer_content = ds[tracer].fillna(0.)*ds.volcello

### Layer thickness and CFC content

In [25]:
ds_layers = xr.Dataset(coords=ds.coords)
ds_layers = ds_layers.assign_coords({k:v for (k,v) in moc_metrics.coords.items() if "moc" in k})

ds_layers["tracer_binned"] = grid.transform(
    tracer_content,
    "Z",
    target=moc_metrics.sigma2_moc_i,
    target_data=grid.interp(ds.sigma2.ffill("z_l", limit=1), "Z"),
    method="conservative",
).rename({"rho2_moc_i":"rho2_moc_l"}).assign_coords({"rho2_moc_l":moc_metrics.rho2_moc_l})

ds_layers["layer_thickness"] = grid.transform(
    ds.thkcello.fillna(0.),
    "Z",
    target=moc_metrics.sigma2_moc_i,
    target_data=grid.interp(ds.sigma2.ffill("z_l", limit=1), "Z"),
    method="conservative",
).rename({"rho2_moc_i":"rho2_moc_l"}).assign_coords({"rho2_moc_l":moc_metrics.rho2_moc_l})

ds_layers.to_netcdf(f"../data/interim/tracer_content_by_layer_{sim}.nc", mode="w")

In [46]:
ds_layered = xr.Dataset()

In [47]:
with ProgressBar():
    ds_layered[f"{tracer}_layer_content_xyint"] = tracer_binned.sum(["xh", "yh"]).compute()*g_per_mol[tracer]*1e-9 # Gg
    ds_layered[f"{tracer}_layer_content_xyint"].attrs = {
        "long_name": f"globally-integrated oceanic {tracer} content by density layer",
        "units": "Gg",
    }

In [48]:
with ProgressBar():
    ds_layered[f"fg{tracer}_layer_xyint"] = xhistogram.xarray.histogram(
        ds.sigma2_surface.rename("sigma2"),
        bins=moc_metrics.rho2_moc_i.values,
        dim=("xh", "yh",),
        weights=ds.areacello*ds[f"fg{tracer}"],
        bin_dim_suffix="_l",
        # TEMPORARY FIX FOR https://github.com/xgcm/xhistogram/issues/16
        block_size=None
    ).compute().groupby("time.year").mean("time")
    ds_layered[f"fg{tracer}_layer_xyint"].attrs = {}

[########################################] | 100% Completed | 79.94 s


In [50]:
ds_layered[f"{tracer}_layer"] = xr.concat(
    [
        (tracer_binned / ds.areacello).fillna(0.).sel(year=slice(1991+years_after, 1997+years_after)).mean("year")
        .expand_dims({"year_start": np.array([1991+years_after])})
        for years_after in [0, 30, 60]
    ],
    dim="year_start"
)
ds_layered[f"{tracer}_layer"].attrs = {
    "long_name": f"vertically-integrated {tracer} concentration",
    "units": "mol/m^2",
    "description": f"time-mean vertically-integrated {tracer} concentration from `year_start` to `year_start + 6` (inclusive)",
}

In [55]:
ds_layered.to_netcdf(f"../data/processed/{tracer}_rho2layer_budget.nc", mode="w")